In [8]:
import pandas as pd
import numpy as np

movies = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')
tags = pd.read_csv('data/tags.csv')

movies.shape[0]

86537

In [9]:
movies_nup = movies.drop_duplicates("title")
movies_nup_with_genres = movies_nup[movies_nup["genres"] != "(no genres listed)"]
movies_nup_with_genres.shape[0] / movies.shape[0]

0.9161976957833066

In [10]:
min_rating_count = 50 # Set to zero to disable filtering

y = ratings.groupby("movieId").count()["rating"] > min_rating_count
ratings_for_famous_movies = y[y].index
movies_famous = movies_nup_with_genres[movies_nup_with_genres["movieId"].isin(ratings_for_famous_movies)]
movies_famous.shape[0] / movies.shape[0]

0.18358621167824168

In [11]:
movies_famous["genres"] = movies_famous["genres"].str.replace("|", " ", regex=False) # "Adventure|Animation..."" -> "Adventure" "Animation"...

tags_for_famous_movies = tags[tags["movieId"].isin(movies_famous["movieId"])]
tags_for_famous_movies = tags_for_famous_movies.dropna(subset=["tag"])

tags_grouped = tags_for_famous_movies.groupby("movieId")["tag"].apply(" ".join)         # All tags for specific movie grouped

movies_famous_with_tags = movies_famous.merge(tags_grouped, on="movieId", how="left")             # column for grouped tags is added to movies_famous

movies_famous_with_tags["tag"] = movies_famous_with_tags["tag"].fillna("")

movies_famous_with_tags["combined_text"] = movies_famous_with_tags["genres"] + " " + movies_famous_with_tags["tag"]
movies_famous_with_tags = movies_famous_with_tags.drop(columns = ["genres", "tag"])



In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


tfidf = TfidfVectorizer(stop_words="english")
feature_matrix = tfidf.fit_transform(movies_famous_with_tags["combined_text"])
similarity_score = cosine_similarity(feature_matrix)
similarity_score

array([[1.        , 0.06024934, 0.01935714, ..., 0.01226428, 0.05679647,
        0.17763386],
       [0.06024934, 1.        , 0.00458008, ..., 0.01967786, 0.01149217,
        0.04952561],
       [0.01935714, 0.00458008, 1.        , ..., 0.0189297 , 0.        ,
        0.00579814],
       ...,
       [0.01226428, 0.01967786, 0.0189297 , ..., 1.        , 0.07202691,
        0.10801082],
       [0.05679647, 0.01149217, 0.        , ..., 0.07202691, 1.        ,
        0.04923022],
       [0.17763386, 0.04952561, 0.00579814, ..., 0.10801082, 0.04923022,
        1.        ]], shape=(15887, 15887))

In [13]:
import numpy as np

def recommend(movie_title):
    match = movies_famous_with_tags[movies_famous_with_tags["title"] == movie_title]
    if match.empty:
       return "Movie not found"
    index = match.index[0]

    similar_movies = sorted(list(enumerate(similarity_score[index])), key=lambda x: x[1], reverse=True)[1:6]
    data = []

    for idx, similarity in similar_movies:
        item = []
        title = movies_famous_with_tags.iloc[idx]["title"]
        similarity_proc = round(float(similarity * 100), 1)      
        item.append(title)
        item.append(similarity_proc)
        data.append(item)
    return data

In [14]:
recommend("Yellow Submarine (1968)")

[['Magical Mystery Tour (1967)', 81.0],
 ['Help! (1965)', 77.7],
 ['Across the Universe (2007)', 70.8],
 ["Hard Day's Night, A (1964)", 66.8],
 ['Imagine: John Lennon (1988)', 60.6]]